# NumPy Project 01 — OrderHub API Health Monitor

**Track:** AI_ML_Series / 01_numpy
**Rule for this project:** pure NumPy. No pandas, no sklearn, no plotting library.
Every answer comes out of arrays.

---

## The brief

You run the platform team at **OrderHub**, an e-commerce backend built from
6 microservices.

Every night a monitoring agent writes one report. For each service it records:

- the **p95 latency** in milliseconds (95% of requests were faster than this)
- the number of **5xx errors** the service returned

Last week's report just landed. It is Monday morning. Your manager wants four
answers before standup:

1. **Which services broke their SLA last week, and on how many days?**
2. **Friday looks terrible. What happened, and which service caused it?**
3. **Which service is the biggest reliability risk going into next week?**
4. **Can we scale the cluster down over the weekend?**

No database. No dashboard. 84 numbers and NumPy.

---

## Why this dataset and not a random one

Every step of this project is a NumPy concept wearing a work shirt:

| The work question | The NumPy concept underneath |
|---|---|
| "What is normal for each service?" | `axis=1` reduction |
| "What is normal for each day?" | `axis=0` reduction |
| "Did we breach our SLA?" | broadcasting + boolean masks |
| "Rank the worst offenders" | `argsort` + fancy indexing |
| "The agent missed a reading on Wednesday" | `np.nan` handling |
| "Put every service on the same scale" | standardisation — what `StandardScaler` does |

If you can answer the manager's four questions, you know NumPy well enough for
the ML track.

## The data

Two 6x7 matrices. **Rows are services, columns are days (Mon -> Sun).**

Keeping that sentence in your head is most of the battle — nearly every bug in
this project will be a row/column mix-up.

### `latency_ms` — p95 latency in milliseconds

| service | Mon | Tue | Wed | Thu | Fri | Sat | Sun |
|---|---|---|---|---|---|---|---|
| auth    |  42 |  45 |  44 |  46 |  51 |  38 |  36 |
| catalog |  58 |  61 |  59 |  63 |  72 |  50 |  47 |
| cart    | 115 | 120 | 118 | 125 | 140 | 102 |  98 |
| search  | 205 | 210 | 215 | 208 | **980** | 195 | 190 |
| payment | 148 | 152 | **NaN** | 150 | 165 | 140 | 135 |
| order   | 175 | 180 | 178 | 185 | 240 | 165 | 160 |

Two things in that table are deliberate, and they are the two data problems you
meet in every real dataset:

- **`search` Friday = 980ms** — 4x its normal value. Is it a broken sensor or a
  real incident? Part 2 decides, and the decision is not "delete it".
- **`payment` Wednesday = NaN** — the monitoring agent restarted during its
  collection window. The reading does not exist. It is not zero.

### `errors_5xx` — count of 5xx responses per day

| service | Mon | Tue | Wed | Thu | Fri | Sat | Sun |
|---|---|---|---|---|---|---|---|
| auth    |  2 |  1 |  3 |  2 |   5 |  1 |  0 |
| catalog |  4 |  3 |  5 |  4 |   9 |  2 |  2 |
| cart    |  8 |  7 |  9 | 10 |  22 |  5 |  4 |
| search  |  6 |  5 |  7 |  6 |  85 |  4 |  3 |
| payment | 11 | 12 | 10 | 13 |  31 |  8 |  7 |
| order   | 14 | 13 | 15 | 16 |  48 | 10 |  9 |

### `sla_ms` — the promise each service made (one per service)

| auth | catalog | cart | search | payment | order |
|---|---|---|---|---|---|
| 50 | 65 | 120 | 200 | 150 | 190 |

Note these are **different per service**. A single number would not need
broadcasting; six numbers against a 6x7 matrix do. That is the point.

### `traffic_k` — platform requests per day, in thousands (one per day)

| Mon | Tue | Wed | Thu | Fri | Sat | Sun |
|---|---|---|---|---|---|---|
| 420 | 455 | 468 | 482 | 610 | 305 | 268 |

Friday was a promo day. Hold that thought — it explains a lot.

## The flow — 12 parts

| # | Part | The question it answers | NumPy you will use |
|---|---|---|---|
| 1 | Load & inspect | What did we actually receive? | `np.array`, `dtype`, `shape`, `ndim`, `size`, `nbytes` |
| 2 | Data quality | One reading is missing, one looks insane | `np.isnan`, `np.where`, `np.nanmean`, `np.median`, `.copy()` vs view |
| 3 | Aggregation & axis | What is normal per service? per day? | `mean/min/max/std`, `axis=0` vs `axis=1`, `argmax`, `percentile` |
| 4 | Broadcasting vs SLA | Who broke their promise? | `reshape(-1, 1)`, broadcasting rules, comparison operators |
| 5 | Boolean masking | How many breaches, and exactly where? | boolean arrays, `sum` on bools, `count_nonzero`, `any`/`all`, `np.where(c, a, b)` |
| 6 | Derived metrics | Errors per million, SLA headroom | element-wise math, row-vector broadcast, `np.round` |
| 7 | Ranking | Rank services worst -> best | `argsort`, fancy indexing, `[::-1]`, top-N |
| 8 | Weekday vs weekend | Can we scale down on Sat/Sun? | 2-D slicing, column ranges |
| 9 | Normalisation | Put every service on one scale | min-max, z-score, `keepdims` |
| 10 | Reshape & assemble | Build the report table | `reshape`, `.T`, `ravel` vs `flatten`, `column_stack` |
| 11 | Relationships | Does traffic drive latency? | `corrcoef`, `cov` |
| 12 | Persist & report | Ship it | `set_printoptions`, `np.save`/`load`, `savetxt` |

Parts 1-2 are plumbing. Parts 3-5 answer question 1. Part 6-7 answer questions
2 and 3. Part 8 answers question 4. Parts 9-12 are the bridge to the ML track.

## How we work

- I give you the code for one part at a time. **You type it** — do not paste.
  The typing is what makes it stick.
- Run each cell and check the output against what I said it should be.
- If a line looks strange, ask before moving on. Understanding beats finishing.
- We add the next part's cells to this notebook only when the current part runs.

The empty cells below are Part 1. Parts 2-12 get added as we go.

---

## Part 1 — Load & inspect

**Goal:** get the four arrays into memory and prove they are the right shape
before we trust a single calculation.

Steps:

- `1.1` import NumPy and set print options so output is readable
- `1.2` the labels — service names and day names
- `1.3` the latency matrix (the NaN goes in here)
- `1.4` the errors matrix
- `1.5` the SLA array and the traffic array
- `1.6` sanity checks — shape, dtype, size, memory

In [1]:
# 1.1  imports and print options
import numpy as np

# make float output readable: 1 decimal, no scientific notation
np.set_printoptions(precision=1, suppress=True)
np.__version__


'2.4.6'

In [2]:
# 1.2  labels: services and days
services = np.array(['auth', 'catalog', 'cart', 'search', 'payment', 'order'])
days     = np.array(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])

services.shape, days.shape


((6,), (7,))

In [8]:
# 1.3  latency_ms  (6 services x 7 days)
latency_ms = np.array([
    [ 42,  45,  44,  46,  51,  38,  36],   # auth
    [ 58,  61,  59,  63,  72,  50,  47],   # catalog
    [115, 120, 118, 125, 140, 102,  98],   # cart
    [205, 210, 215, 208, 980, 195, 190],   # search   <- Friday spike
    [148, 152, np.nan, 150, 165, 140, 135],  # payment <- Wednesday missing
    [175, 180, 178, 185, 240, 165, 160],   # order
], dtype=float)

latency_ms


array([[ 42.,  45.,  44.,  46.,  51.,  38.,  36.],
       [ 58.,  61.,  59.,  63.,  72.,  50.,  47.],
       [115., 120., 118., 125., 140., 102.,  98.],
       [205., 210., 215., 208., 980., 195., 190.],
       [148., 152.,  nan, 150., 165., 140., 135.],
       [175., 180., 178., 185., 240., 165., 160.]])

In [3]:
# 1.4  errors_5xx  (6 services x 7 days)'
errors_5xx = np.array([
    [ 2,  1,  3,  2,  5,  1,  0],   # auth
    [ 4,  3,  5,  4,  9,  2,  2],   # catalog
    [ 8,  7,  9, 10, 22,  5,  4],   # cart
    [ 6,  5,  7,  6, 85,  4,  3],   # search
    [11, 12, 10, 13, 31,  8,  7],   # payment
    [14, 13, 15, 16, 48, 10,  9],   # order
])

errors_5xx


array([[ 2,  1,  3,  2,  5,  1,  0],
       [ 4,  3,  5,  4,  9,  2,  2],
       [ 8,  7,  9, 10, 22,  5,  4],
       [ 6,  5,  7,  6, 85,  4,  3],
       [11, 12, 10, 13, 31,  8,  7],
       [14, 13, 15, 16, 48, 10,  9]])

In [6]:
# 1.5  sla_ms (per service) and traffic_k (per day)
# one SLA per service  -> lines up with ROWS
sla_ms = np.array([50, 65, 120, 200, 150, 190])

# platform traffic in thousands, one per day -> lines up with COLUMNS
traffic_k = np.array([420, 455, 468, 482, 610, 305, 268])

sla_ms.shape, traffic_k.shape


((6,), (7,))

In [9]:
# 1.6  sanity checks
print("latency  shape:", latency_ms.shape, "dtype:", latency_ms.dtype)
print("errors   shape:", errors_5xx.shape, "dtype:", errors_5xx.dtype)
print("ndim:", latency_ms.ndim, "| size:", latency_ms.size, "| bytes:", latency_ms.nbytes)

# fail loudly now rather than get a wrong answer in Part 4
assert latency_ms.shape == errors_5xx.shape == (len(services), len(days))
assert sla_ms.shape[0] == len(services)
assert traffic_k.shape[0] == len(days)
print("\nall shape checks passed")


latency  shape: (6, 7) dtype: float64
errors   shape: (6, 7) dtype: int64
ndim: 2 | size: 42 | bytes: 336

all shape checks passed


### Part 1 checkpoint

Before moving on, you should be able to answer these without scrolling up:

1. Why is `latency_ms.dtype` `float64` when almost every value is a whole number?
2. What does `latency_ms.shape` return, and which number is services?
3. `sla_ms` has shape `(6,)` and `traffic_k` has shape `(7,)`. Why do those two
   sizes differ, and what does each one line up with?

---

*Parts 2-12 will be appended here as we build them.*


## Part 2 — Data quality: the missing reading and the 980

**Goal:** get to a matrix we can trust, and make a defensible decision about
each of the two problems.

The two problems look similar — both are "weird values" — but they need
opposite treatment:

| | `payment` Wed = NaN | `search` Fri = 980 |
|---|---|---|
| What it is | data that does not exist | data that does exist |
| Cause | monitoring agent restarted | a real production incident |
| Right action | **replace it** with an estimate | **keep it**, and flag it |
| Wrong action | fill with `0` | delete it |

Deleting the 980 would erase the exact event your manager is asking about in
question 2. An outlier is not automatically an error.

Steps:

- `2.1` find the missing reading — and watch one NaN poison every calculation
- `2.2` why filling with `0` is the wrong fix
- `2.3` copy vs view — why we work on a copy
- `2.4` impute the missing value from the service's own week
- `2.5` hunt the outlier with the textbook z-score test — it **fails**
- `2.6` why it fails
- `2.7` the median/MAD test — it catches it
- `2.8` the decision: flag, do not delete

In [10]:
# 2.1  find the missing reading
print("mean of the whole matrix:", latency_ms.mean())
print("missing readings:", np.isnan(latency_ms).sum())

rows, cols = np.where(np.isnan(latency_ms))
print("row(s):", rows, " col(s):", cols)
print("->", services[rows[0]], "on", days[cols[0]])


mean of the whole matrix: nan
missing readings: 1
row(s): [4]  col(s): [2]
-> payment on Wed


In [11]:
# 2.2  why filling with 0 is wrong
payment = latency_ms[4]

print("payment week       :", payment)
print("nanmean (skip it)  :", round(np.nanmean(payment), 2))
print("filled with 0      :", round(np.nan_to_num(payment).mean(), 2))


payment week       : [148. 152.  nan 150. 165. 140. 135.]
nanmean (skip it)  : 148.33
filled with 0      : 127.14


In [17]:
# 2.3  copy vs view
view = latency_ms[0]        # a slice is a VIEW, shared memory
print("View", view[0])
view[0] = 999
print("View", view[0])
print("original after touching the view:", latency_ms[0, 0])

latency_ms[0, 0] = 42       # put it back
clean = latency_ms.copy()   # .copy() = independent memory
clean[0, 0] = 999
print("copy:", clean[0, 0], "| original safe:", latency_ms[0, 0])
clean[0, 0] = 42


View 42.0
View 999.0
original after touching the view: 999.0
copy: 999.0 | original safe: 42.0


In [20]:
# 2.4  impute the missing value
#
# GOAL: payment has no Wednesday number. Estimate it from payment's own week.
# We fill on a COPY so the raw measurements stay provable.

# .copy() = new memory. Without it we'd overwrite the raw data (see 2.3).
clean = latency_ms.copy()

# rows/cols came from 2.1 and are ARRAYS: rows=[4], cols=[2].
# latency_ms[rows] -> shape (1, 7), still 2-D, because we indexed with an array.
# nanmean(axis=1)  -> collapse the 7 days into 1 number per row, skipping NaN.
#                     Divides by 6, not 7. Result: [148.33], shape (1,).
# clean[rows, cols] pairs rows[0] with cols[0] -> writes to cell (4,2) only.
clean[rows, cols] = np.nanmean(latency_ms[rows], axis=1)

print("imputed value        :", round(clean[4, 2], 1))   # 148.3
print("any NaN left in clean:", np.isnan(clean).any())   # False
print("original still NaN   :", np.isnan(latency_ms[4, 2]))  # True - raw is safe
print("clean matrix mean    :", round(clean.mean(), 1))  # 144.2 (was nan)



imputed value        : 148.3
any NaN left in clean: False
original still NaN   : True
clean matrix mean    : 144.2


In [21]:
# 2.5  outlier hunt, attempt 1: the z-score test
search = clean[3]
z = (search - search.mean()) / search.std()

print("search week:", search)
print("z-scores   :", np.round(z, 2))
print("max z      :", round(z.max(), 2))
print("flagged at the usual threshold of 3?", bool(z.max() > 3))


search week: [205. 210. 215. 208. 980. 195. 190.]
z-scores   : [-0.4 -0.4 -0.4 -0.4  2.5 -0.4 -0.5]
max z      : 2.45
flagged at the usual threshold of 3? False


In [22]:
# 2.6  why the z-score test failed
print("mean:", round(search.mean(), 1), " <- dragged UP by the 980")
print("std :", round(search.std(), 1), " <- inflated BY the 980")

without_fri = np.delete(search, 4)
print("\nwithout Friday -> mean:", round(without_fri.mean(), 1),
      " std:", round(without_fri.std(), 1))
print("z of 980 against THAT std:",
      round((980 - without_fri.mean()) / without_fri.std(), 1))


mean: 314.7  <- dragged UP by the 980
std : 271.7  <- inflated BY the 980

without Friday -> mean: 203.8  std: 8.7
z of 980 against THAT std: 89.5


In [23]:
# 2.7  outlier hunt, attempt 2: median + MAD
med = np.median(search)
mad = np.median(np.abs(search - med))     # median absolute deviation
modified_z = 0.6745 * (search - med) / mad

print("median:", med, " MAD:", mad)
print("modified z:", np.round(modified_z, 1))
print("max:", round(modified_z.max(), 1),
      "-> flagged at threshold 3.5?", bool(modified_z.max() > 3.5))


median: 208.0  MAD: 7.0
modified z: [-0.3  0.2  0.7  0.  74.4 -1.3 -1.7]
max: 74.4 -> flagged at threshold 3.5? True


In [24]:
# 2.8  the decision: flag, do not delete
med_r = np.median(clean, axis=1, keepdims=True)
mad_r = np.median(np.abs(clean - med_r), axis=1, keepdims=True)
mod_z = 0.6745 * (clean - med_r) / mad_r

incident = mod_z > 3.5
print("incident cells:", incident.sum())
for i, j in zip(*np.where(incident)):
    print(f"  {services[i]:8s} {days[j]}  {clean[i, j]:6.0f} ms   modified z = {mod_z[i, j]:.1f}")

print("\nrows kept:", clean.shape, "- nothing deleted")


incident cells: 2
  search   Fri     980 ms   modified z = 74.4
  order    Fri     240 ms   modified z = 6.0

rows kept: (6, 7) - nothing deleted


### Part 2 checkpoint

1. `np.where(np.isnan(latency_ms))` returns a **tuple of two arrays**, not one.
   Why two? What would it return for a 3-D array?
2. We imputed with `np.nanmean(latency_ms[r], axis=1)` — the mean of *that
   service's own week*. Why not the mean of the whole matrix?
3. The z-score test said `2.45` for a value that is 4x normal. Explain, in one
   sentence, how the outlier hid from the test that was hunting it.
4. `clean = latency_ms.copy()` — what would have gone wrong with
   `clean = latency_ms`?

---

*Parts 3-12 will be appended here as we build them.*